# MiniGPT with the modern block: follow along

This notebook goes with my post [MiniGPT (Part 5)](https://haddley.github.io/posts/minigpt4/). It builds the 2017 block and Llama's modern block, counts what each change costs, shows rotary positions (RoPE) at work, trains the modern block, and writes a story.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part5-modern-block")

working in part5-modern-block


## 1. Six machines, and what each change costs

The new block, then each change turned back off on its own. Every machine has the same 8,192 token cards.

In [2]:
import mlx.core as mx
from mlx.utils import tree_flatten
from model_llama import Config, MiniLlama

variants = {
    "new block, all four changes": {},
    "... but the old MLP":        {"mlp": "gelu"},
    "... but full keys and values": {"n_kv_heads": 6},
    "... but the old normalise":  {"norm": "layer"},
    "... but position cards":     {"pos": "learned"},
}
for name, change in variants.items():
    m = MiniLlama(Config(vocab_size=8192, **change))
    mx.eval(m.parameters())
    print(f"{name:32} {sum(v.size for _, v in tree_flatten(m.parameters())):>12,} numbers")

new block, all four changes        12,587,904 numbers
... but the old MLP                12,599,424 numbers
... but full keys and values       13,767,552 numbers
... but the old normalise          12,592,896 numbers
... but position cards             12,686,208 numbers


## 2. RoPE: the match only feels the distance

Turn the same query card and key card for two pairs of positions with the same gap: (7, 5) and (107, 105). With RoPE, the two matches come out the same.

In [3]:
import mlx.nn as nn
rope = nn.RoPE(64, traditional=False, base=10000)
mx.random.seed(0)
q = mx.random.normal((1, 1, 1, 64))
k = mx.random.normal((1, 1, 1, 64))

def turned(card, position):
    return rope(card, offset=position)

for qp, kp in ((7, 5), (107, 105), (7, 3)):
    match = (turned(q, qp) * turned(k, kp)).sum().item()
    print(f"query at {qp:3}, key at {kp:3} (gap {qp - kp}): match {match:8.4f}")

query at   7, key at   5 (gap 2): match   2.5921
query at 107, key at 105 (gap 2): match   2.5921
query at   7, key at   3 (gap 4): match   8.3145


## 3. Train the new block

3,000 steps on Part 3's stories, checking bits per byte every 300 steps. On my Mac Studio, 3,000 steps took about 7.5 minutes. `run_ablation.sh` trains all five variants, about 40 minutes in all.

In [4]:
run("train_llama.py", "--tag", "modern", "--iters", "3000", "--eval-interval", "300")

tag modern  norm rms  pos rope  mlp swiglu  kv_heads 2  params 12,587,904
step     0  train 9.5006  val 9.5095  val bpb 3.3420
step   300  train 2.7455  val 2.8170  val bpb 0.9900
step   600  train 2.3670  val 2.4277  val bpb 0.8532
step   900  train 2.1283  val 2.2600  val bpb 0.7943
step  1200  train 2.0025  val 2.1244  val bpb 0.7466
step  1500  train 1.9027  val 2.0651  val bpb 0.7258
step  1800  train 1.8202  val 2.0249  val bpb 0.7116
step  2100  train 1.7619  val 1.9480  val bpb 0.6846
step  2400  train 1.7330  val 1.9121  val bpb 0.6720
step  2700  train 1.6895  val 1.9152  val bpb 0.6731
step  3000  train 1.6600  val 1.9112  val bpb 0.6717
done in 6.95 min  best val 1.9112  peak 3.57 GB



## 4. What it writes

In [5]:
run("generate_llama.py", "--tag", "modern", "--prompt", "Once upon a time", "--max-new-tokens", "150")

Once upon a time, there was a boy named Tim. Tim loved to play with his toy car. One day, he found a big knot in his room. He was very happy and wanted to show it to his mom.
Tim ran outside to look for the knot. He saw his friend, Sue, sitting on the ground. Tim said, "Sue, can I use the knot?" Sue said, "Yes, but first, we need to be careful." So they closed the knot and took the knot in.
When they opened the knot, they saw a big bird. The bird said, "I can help you, Tim and Sue!" Tim and Sue were excited to help the bird. They ran to the knot and got the knot. The

